# BM25关键词匹配  
需要预设schema、index，在schema中加入bm25的function。  
schema中的bm25 function，能够自动在指定field中生成对应的稀疏向量。传入时便只需要在待转换field中传入文本。  
- 待转换文本field必须启动`enable_analyzer`，才能正常使用bm25 function。可以配合加上`analyzer_params = {"type": "chinese"}`，指定文本分析器。  
- 自动转换的稀疏向量只能用于检索，不能输出（缺陷：无法用output_fields提取出来直接使用）。

In [1]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

In [2]:
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

In [3]:
schema = client.create_schema()
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
# 待转换文本列
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1024, enable_analyzer=True, analyzer_params={"type": "chinese"})
# 保存自动转换的稀疏向量列
schema.add_field(field_name="bm25_sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1024, 'enable_analyzer': True, 'analyzer_params': '{"type":"chinese"}'}}, {'name': 'bm25_sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [4]:
# 定义bm25的向量转换函数
bm25_function = Function(
    name="text_bm25_emb",
    input_field_names=["text"], # 待转换列
    output_field_names=["bm25_sparse"], # 保存自动转换的稀疏向量列
    function_type=FunctionType.BM25, # 函数类型为bm25
)
schema.add_function(bm25_function)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1024, 'enable_analyzer': True, 'analyzer_params': '{"type":"chinese"}'}}, {'name': 'bm25_sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_output': True}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'text_bm25_emb', 'description': '', 'type': <FunctionType.BM25: 1>, 'input_field_names': ['text'], 'output_field_names': ['bm25_sparse'], 'params': {}}]}

In [5]:
# 配置索引（详见milvus数据管理文档）
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="bm25_sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.2,
        "bm25_b": 0.75
    }
)

In [6]:
client.create_collection(
    collection_name="bm25_collection",
    schema=schema,
    index_params=index_params,
    consistency_level="Strong"
)
client.load_collection(collection_name="bm25_collection")

插入文本数据便会自动转换为稀疏向量，并保存到bm25_sparse列中。

In [7]:
res= client.insert(
    collection_name="bm25_collection",
    data=[
        {"text": "这是一个文档"},
        {"text": "这是另一个文档"},
    ]
)
print(res)

{'insert_count': 2, 'ids': [463638649837937828, 463638649837937829]}


检索匹配目标列时，便会按照bm25算法进行匹配  
- **请勿返回bm25_sparse列，自动转换的稀疏向量不支持检索返回**

In [8]:
res = client.search(
    collection_name="bm25_collection",
    data=["这是另外的文档"],
    anns_field="bm25_sparse",
    output_fields=["text"],
    limit=2
)
print(res)

data: [[{'id': 463638649837937828, 'distance': 0.975205659866333, 'entity': {'text': '这是一个文档'}}, {'id': 463638649837937829, 'distance': 0.16540512442588806, 'entity': {'text': '这是另一个文档'}}]]


# BGE-M3语义匹配（基于HuggingFace TEI）  
* 首先使用HuggingFace官方方法，将TEI作为独立服务部署（保障灵活性与控制权）。使用docker部署并保存TEI服务端点。参考https://huggingface.co/docs/text-embeddings-inference/en/quick_tour#deploy  
    更多TEI部署可参考https://wcneb3yvewfa.feishu.cn/docx/HLMHdBIbuoZaiOx9FaGceCWOnXf  
* 待转换文本列需要控制字数。bge-m3最大token为8194，建议varchar的max_length同样为8194。  
* bge-m3模型输出为1024维向量，建议float_vector的dim为1024。  
* 定义嵌入函数时，通常参数包括：
    * **provider**：必须。TEI方式为TEI
    * **endpoint**：必须。TEI服务port，例如http://localhost:8080。当milvus在docker中运行时，想访问宿主机的端口服务，需要使用http://host.docker.internal:8080
    * **truncate**：可选。截断超出模型嵌入长度的文本。"true"/"false"（default）
    * **truncation_direction**：可选。截断方向。"Left"/"Right"（default）
    * max_client_batch_size ：Milvus客户端发送到TEI的最大批量大小。默认为32。
    * prompt_name：高级可选。用于某些需要特定提示格式的模型。"your_prompt_key"
    * ingestion_prompt：高级可选。指定在数据插入（摄取）阶段使用的提示。"passage："
    * search_prompt：高级可选。指定在搜索阶段使用的提示。"query: "
* 自动转换的向量同样无法在检索时导出。

In [4]:
from pymilvus import MilvusClient, DataType, Function, FunctionType, CollectionSchema, FieldSchema

In [24]:
client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

In [29]:
schema = MilvusClient.create_schema()
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8194) # 注意文本长度
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024) # 注意向量维度

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 8194}}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [30]:
# 定义embedding函数
text_embedding_function = Function(
    name="bge-m3-func",
    function_type=FunctionType.TEXTEMBEDDING,
    input_field_names=["text"],
    output_field_names=["dense"],
    params={
        "provider": "TEI",
        "endpoint": "http://host.docker.internal:8080",
        "truncate": "true",
        "truncation_direction": "Right",
        # "max_client_batch_size": 64,
    }
)
schema.add_function(text_embedding_function)

{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 8194}}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}, 'is_function_output': True}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'bge-m3-func', 'description': '', 'type': <FunctionType.TEXTEMBEDDING: 2>, 'input_field_names': ['text'], 'output_field_names': ['dense'], 'params': {'provider': 'TEI', 'endpoint': 'http://host.docker.internal:8080', 'truncate': 'true', 'truncation_direction': 'Right'}}]}

In [31]:
index_params=client.prepare_index_params()
index_params.add_index(
    field_name="dense",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

In [32]:
client.create_collection(
    collection_name="bge_m3_test",
    schema=schema,
    index_params=index_params,
    consistency_level="Strong",
)
client.load_collection(collection_name="bge_m3_test")

In [35]:
res = client.insert(
    collection_name='bge_m3_test',
    data=[{'text': 'Milvus simplifies semantic search through embeddings.'},
    {'text': 'Vector embeddings convert text into searchable numeric data.'},
    {'text': 'Semantic search helps users find relevant information quickly.'}]
)
print(res)

{'insert_count': 3, 'ids': [463645537557969102, 463645537557969103, 463645537557969104]}


In [36]:
# 检索向量
results = client.search(
    collection_name='bge_m3_test', 
    data=['How does Milvus handle semantic search?'], # 可以直接用原始文本检索，也可以使用嵌入向量
    anns_field='dense',
    limit=2,
    output_fields=['text'],
)

print(results)

data: [[{'id': 463645537557969102, 'distance': 0.7975181937217712, 'entity': {'text': 'Milvus simplifies semantic search through embeddings.'}}, {'id': 463645537557969104, 'distance': 0.5966318249702454, 'entity': {'text': 'Semantic search helps users find relevant information quickly.'}}]]
